# 混合专家模型 (MoE) 与智能体 (AI Agents) 企业笔试手撕通关宝典
> **面向对象**：互联网大厂/AI 独角兽企业 MoE 架构研发、智能体 Agent 研发、DeepSeek 核心演进面试手撕  
> **核心涵盖**：Top-K 稀疏门控、辅助负载均衡损失、DeepSeekMoE 共享专家隔离、免辅助损失动态偏置调节、MoE 分桶分发、ReAct 核心状态机、Tool Call 注册分发器、Multi-Agent 协同反思闭环  
> **设计准则**：纯 PyTorch / Python 逐行手推，彻底解构专家稀疏激活与智能体决策拓扑。

---
### 核心模块速览
1. **模块一**：Top-K 稀疏门控路由 (Top-K Gating Router) 纯手撕 (带 Softmax 局部重归一化)
2. **模块二**：传统辅助负载均衡损失 (Auxiliary Load Balancing Loss) 手撕
3. **模块三**：DeepSeekMoE 细粒度路由与共享专家 (Shared Experts) 隔离手撕
4. **模块四**：DeepSeek-V3 免辅助损失动态偏置调节路由机制 (Aux-Loss-Free Dynamic Bias)
5. **模块五**：极简 MoE 层前向计算与专家分发调度 (Dispatch & Combine) 手撕
6. **模块六**：ReAct 智能体核心状态机手撕 (Thought-Action-Observation 闭环)
7. **模块七**：工具调用分发器 (Tool Call Dispatcher) 与 JSON 参数解析手撕
8. **模块八**：双智能体对齐协作架构手撕 (Coder + Critic Reviewer 辩论反思机制)

---
## 模块一：Top-K 稀疏门控路由 (Top-K Gating Router) 纯手撕

### 【笔试考点与推导】
1. **打分计算**：$H(x) = x W_g$；
2. **Top-K 提取**：获取分数最高的前 $k$ 个专家下标与打分；
3. **局部 Softmax 重归一化**：
   $$P(x)_i = \frac{\exp(H(x)_i)}{\sum_{j \in \text{TopK}} \exp(H(x)_j)}$$
   只有被选中的 $k$ 个专家的权重之和严格等于 1。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TopKGatingRouter(nn.Module):
    def __init__(self, d_model, num_experts, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.w_gate = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x):
        """
        x: (B, d_model)
        """
        logits = self.w_gate(x) # (B, num_experts)
        
        # 挑选 Top-K 专家
        topk_logits, topk_indices = torch.topk(logits, self.top_k, dim=-1) # (B, top_k)
        
        # 仅对 Top-K 的得分做局部 Softmax 重归一化
        topk_weights = F.softmax(topk_logits, dim=-1) # (B, top_k)
        return topk_weights, topk_indices, logits

# 测试门控
router = TopKGatingRouter(d_model=16, num_experts=8, top_k=2)
dummy_x = torch.randn(4, 16) # B=4
weights, indices, logits = router(dummy_x)
print("选中专家索引 (B, TopK):\n", indices.numpy())
print("选中专家归一化权重 (总和=1):\n", weights.detach().numpy())
assert torch.allclose(weights.sum(dim=-1), torch.ones(4))
print(">>> Top-K 稀疏门控验证成功！")

---
## 模块二：传统辅助负载均衡损失 (Auxiliary Load Balancing Loss) 手撕

### 【笔试高频考点】
- **Switch Transformer / GShard 公式**：
  $$\mathcal{L}_{\text{aux}} = \alpha \cdot N \sum_{i=1}^N f_i \cdot P_i$$
  - $N$：专家总数；
  - $f_i = \frac{1}{B} \sum_{b=1}^B \mathbb{I}(\text{专家 } i \text{ 在样本 } b \text{ 的 Top-K 中})$：实际分配给专家 $i$ 的 Token 比例（离散不可导）；
  - $P_i = \frac{1}{B} \sum_{b=1}^B \text{softmax}(\text{logits}_b)_i$：门控网络对专家 $i$ 的平均预测概率（平滑可导）；
  - 乘积形式确保梯度能够反向传播调节路由器，驱使 $P_i \approx 1/N$。

In [ ]:
def compute_auxiliary_loss(logits, indices, num_experts, alpha=0.01):
    """
    计算 Switch / GShard 经典辅助负载均衡损失
    logits: (B, num_experts)
    indices: (B, top_k)
    """
    B = logits.size(0)
    top_k = indices.size(1)
    
    # 1. P_i: 全局 Softmax 平均概率 (可导)
    probs = F.softmax(logits, dim=-1)                      # (B, N)
    P = probs.mean(dim=0)                                  # (N,)
    
    # 2. f_i: 实际被选中的频次占比 (离散)
    mask = torch.zeros((B, num_experts), device=logits.device)
    mask.scatter_(dim=1, index=indices, value=1.0)
    f = mask.mean(dim=0) / top_k                           # 归一化频率 (N,)
    
    # 3. 损失计算: alpha * N * sum(f_i * P_i)
    aux_loss = alpha * num_experts * torch.sum(f * P)
    return aux_loss

aux_l = compute_auxiliary_loss(logits, indices, num_experts=8)
print("传统辅助负载均衡 Loss 标量:", aux_l.item())
assert aux_l.item() >= 0.0
print(">>> 辅助平衡损失计算验证通过！")

---
## 模块三：DeepSeekMoE 细粒度路由与共享专家 (Shared Experts) 隔离手撕

### 【DeepSeek 核心面试考点】
1. **细粒度专家切分 (Fine-Grained)**：将常规专家切分成更小的碎片（如 $1/4$ 尺寸），增加专家组合灵活性；
2. **共享专家隔离 (Shared Experts)**：指定若干专家**恒定被激活（无条件计算）**，负责承载公共常识；其余专家按稀疏路由竞争，负责垂直领域知识，避免通用知识在各专家间重复冗余记忆。

In [ ]:
class DeepSeekMoELayer(nn.Module):
    def __init__(self, d_model, num_shared_experts=1, num_routed_experts=6, top_k=2):
        super().__init__()
        self.d_model = d_model
        self.top_k = top_k
        
        # 共享专家 (始终激活)
        self.shared_experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Linear(d_model * 2, d_model))
            for _ in range(num_shared_experts)
        ])
        
        # 路由专家
        self.routed_experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Linear(d_model * 2, d_model))
            for _ in range(num_routed_experts)
        ])
        
        self.router = TopKGatingRouter(d_model, num_routed_experts, top_k)

    def forward(self, x):
        # 1. 计算共享专家输出 (全量累加)
        shared_out = sum(expert(x) for expert in self.shared_experts)
        
        # 2. 路由专家稀疏加权计算
        weights, indices, _ = self.router(x)
        routed_out = torch.zeros_like(x)
        
        for k in range(self.top_k):
            expert_idx = indices[:, k]                      # (B,)
            w = weights[:, k].unsqueeze(-1)                 # (B, 1)
            # 对当前槽位的专家执行前向
            # 为演示清晰遍历 batch
            for b in range(x.size(0)):
                idx = expert_idx[b].item()
                routed_out[b] += w[b] * self.routed_experts[idx](x[b:b+1]).squeeze(0)
                
        # 3. 融合共享专家与路由专家输出
        return shared_out + routed_out

ds_moe = DeepSeekMoELayer(d_model=16, num_shared_experts=2, num_routed_experts=4, top_k=2)
moe_out = ds_moe(dummy_x)
print("DeepSeekMoE 输出形状:", moe_out.shape)
assert moe_out.shape == dummy_x.shape
print(">>> DeepSeekMoE 共享专家与细粒度路由验证成功！")

---
## 模块四：DeepSeek-V3 免辅助损失动态偏置调节路由机制 (Aux-Loss-Free Dynamic Bias)

### 【笔试顶级前沿考点】
1. **为什么抛弃辅助损失？**：辅助损失强行让梯度偏向均衡，会严重损害模型的语言建模能力；
2. **动态偏置破局**：门控打分加上一个动态偏置项 $b_i$：
   $$\text{Score}_i = s_i + b_i$$
   - 在每个 Step 监控各专家的实际负载 $C_i$ 与目标均值 $\bar{C}$；
   - 若某专家过载（$C_i > \bar{C}$），调小其偏置 $b_i \leftarrow b_i - \gamma$；
   - 若某专家闲置（$C_i < \bar{C}$），调大其偏置 $b_i \leftarrow b_i + \gamma$；
   - **偏置不参与梯度反向传播，完全通过推理/运行时反馈动态调节**！

In [ ]:
class DynamicBiasRouter:
    def __init__(self, num_experts, top_k=2, gamma=0.01):
        self.num_experts = num_experts
        self.top_k = top_k
        self.gamma = gamma
        # 动态偏置向量 (非梯度参数)
        self.bias = torch.zeros(num_experts)

    def route_and_update(self, raw_logits):
        """
        raw_logits: (B, num_experts)
        """
        B = raw_logits.size(0)
        # 叠加动态偏置后挑选 Top-K
        effective_logits = raw_logits + self.bias
        _, indices = torch.topk(effective_logits, self.top_k, dim=-1)
        
        # 统计实际各专家的装载数 (Token 计数)
        counts = torch.bincount(indices.view(-1), minlength=self.num_experts).float()
        target_count = (B * self.top_k) / self.num_experts
        
        # 动态调节偏置: 过载扣分，欠载加分
        diff = target_count - counts # 欠载为正，过载为负
        self.bias += self.gamma * torch.sign(diff)
        
        return indices, counts

# 测试动态偏置调节
bias_router = DynamicBiasRouter(num_experts=4, top_k=1, gamma=0.5)
# 模拟 0 号专家一开始拥有极高的初始分数 (容易造成过载崩溃)
heavy_logits = torch.tensor([[10.0, 1.0, 1.0, 1.0]] * 10)
indices_step1, counts_1 = bias_router.route_and_update(heavy_logits)
print("初始步 0 号专家负载超载严重:", counts_1.numpy())

# 经历 10 轮偏置降温扣分
for _ in range(10):
    bias_router.route_and_update(heavy_logits)
print("调节后动态偏置向量 (0号专家已被狠狠压制):", bias_router.bias.numpy())
assert bias_router.bias[0] < 0.0
print(">>> DeepSeek-V3 免辅助损失动态偏置调节验证通过！")

---
## 模块五：极简 MoE 层前向计算与专家分发调度 (Dispatch & Combine) 手撕

### 【笔试考点】
- **分桶执行调度 (Bucket Dispatch)**：将分配给同一专家的 Token 搜集打包为一个批次批量送入 FFN，计算完成后按权重 `combine` 还原回原始 Token 顺序。

In [ ]:
def simple_moe_dispatch_combine(x, experts, router):
    """
    纯张量分桶前向模拟
    """
    B, D = x.shape
    weights, indices, _ = router(x)
    final_output = torch.zeros_like(x)
    
    # 遍历每个专家，搜集被分配到该专家的 token
    for exp_id, expert in enumerate(experts):
        # 找到被选中的样本位置
        for k in range(router.top_k):
            matched_mask = (indices[:, k] == exp_id)
            if matched_mask.any():
                selected_x = x[matched_mask]
                selected_w = weights[matched_mask, k].unsqueeze(-1)
                exp_res = expert(selected_x)
                final_output[matched_mask] += selected_w * exp_res
                
    return final_output

dummy_experts = nn.ModuleList([nn.Linear(16, 16) for _ in range(8)])
out_dispatched = simple_moe_dispatch_combine(dummy_x, dummy_experts, router)
print("分桶调度 MoE 输出形状:", out_dispatched.shape)
assert out_dispatched.shape == (4, 16)
print(">>> 分发与聚合流程调度成功！")

---
## 模块六：ReAct 智能体核心状态机手撕 (Thought-Action-Observation 闭环)

### 【Agent 核心必考考点】
- **ReAct 范式**：
  $$\text{Thought} \to \text{Action}[\text{Tool}(Args)] \to \text{Observation} \to \text{Thought} \dots \to \text{Final Answer}$$
- 状态机通过文本正则拦截 `Action: tool_name[args]`，执行外部真实工具后，将返回值作为 `Observation` 注入上下文中，驱动下一轮推理。

In [ ]:
import re

class ReActAgent:
    def __init__(self, tools_dict):
        self.tools = tools_dict
        self.max_steps = 3

    def step_reasoning(self, history):
        """模拟大模型输出 Thought 与 Action"""
        if "Observation" not in history:
            return "Thought: 我需要计算北京的天气。\nAction: get_weather['Beijing']"
        else:
            return "Thought: 我已拿到天气数据。\nFinal Answer: 北京当前天气晴朗，气温 25 度。"

    def run(self, user_query):
        history = f"Question: {user_query}\n"
        print(f"用户提问: {user_query}")
        
        for step in range(self.max_steps):
            llm_reply = self.step_reasoning(history)
            history += llm_reply + "\n"
            print(f"\n[Step {step+1}] LLM 吐出:\n{llm_reply}")
            
            if "Final Answer:" in llm_reply:
                return llm_reply.split("Final Answer:")[-1].strip()
                
            # 解析 Action: tool[arg]
            match = re.search(r"Action:\s*(\w+)\[\'([^\]]+)\'\]", llm_reply)
            if match:
                tool_name, arg = match.group(1), match.group(2)
                tool_fn = self.tools.get(tool_name)
                obs = tool_fn(arg) if tool_fn else "工具不存在"
                history += f"Observation: {obs}\n"
                print(f"[环境反馈] Observation: {obs}")
                
        return "超过最大步数"

# 测试 ReAct
tools = {"get_weather": lambda city: f"{city} 晴, 25°C"}
agent = ReActAgent(tools)
final_ans = agent.run("北京今天天气怎么样？")
print("\n>>> Agent 最终答复:", final_ans)
assert "25" in final_ans

---
## 模块七：工具调用分发器 (Tool Call Dispatcher) 与 JSON Schema 解析

### 【笔试考点】
- 大模型原生 Tool Call 协议：接收形如 `{"name": "func", "arguments": "{...}"}` 的 JSON 对象，校验参数类型后通过 Python 反射 `getattr` 执行。

In [ ]:
class ToolDispatcher:
    def __init__(self):
        self.registry = {}

    def register(self, name):
        def decorator(func):
            self.registry[name] = func
            return func
        return decorator

    def dispatch(self, tool_call_dict):
        func_name = tool_call_dict.get("name")
        args = tool_call_dict.get("arguments", {})
        if func_name not in self.registry:
            raise ValueError(f"未注册的工具: {func_name}")
        return self.registry[func_name](**args)

dispatcher = ToolDispatcher()

@dispatcher.register("calculate_salary")
def calculate_salary(base, bonus):
    return base + bonus

res = dispatcher.dispatch({"name": "calculate_salary", "arguments": {"base": 20000, "bonus": 5000}})
print("Tool Dispatcher 动态调用执行结果:", res)
assert res == 25000
print(">>> 工具分发注册器验证成功！")

---
## 模块八：双智能体对齐协作架构手撕 (Coder + Critic Reviewer 辩论反思)

### 【笔试考点】
- **多智能体架构**：单 Agent 容易产生盲目幻觉。引入 Critic / Reviewer 角色对初版代码进行审查纠错，形成自我反思闭环（Self-Refine）。

In [ ]:
class MultiAgentCollaboration:
    def __init__(self):
        pass

    def coder_agent(self, task, critique=None):
        if critique is None:
            return "def add(a, b): return a - b" # 故意包含 Bug 的初版
        else:
            return "def add(a, b): return a + b" # 根据反馈修复后的最终版

    def critic_agent(self, code_str):
        if "-" in code_str:
            return "FAIL: 加法函数内部写成了减法，请修复！"
        return "PASS: 代码审查通过无误。"

    def run_debate(self, task):
        print(f"任务启动: {task}")
        # 第一轮: Coder 生成
        draft = self.coder_agent(task)
        print("Coder 生成初版:", draft)
        
        # Critic 评审
        feedback = self.critic_agent(draft)
        print("Critic 评审意见:", feedback)
        assert "FAIL" in feedback
        
        # 第二轮: 根据批评意见修正
        final_code = self.coder_agent(task, critique=feedback)
        print("Coder 修复版代码:", final_code)
        feedback2 = self.critic_agent(final_code)
        print("Critic 二次复审:", feedback2)
        assert "PASS" in feedback2
        return final_code

collab = MultiAgentCollaboration()
fixed = collab.run_debate("实现加法函数")
print(">>> 双智能体审查反思回路验证成功！")

---
## 企业笔试手撕核心口诀与雷区速记卡

```
1. Top-K 门控归一: 挑选前 k 个后必须做局部 softmax 重归一化，保证权重和恒等于 1。
2. 传统平衡损失: alpha * N * sum(f_i * P_i)，用可导的全局预测概率 P 引导不可导的频次 f。
3. DeepSeekMoE 共享: 细粒度切碎专家，常识知识交由 Shared Experts 恒定无条件激活保底。
4. 动态偏置免损失: 运行时过载扣分欠载加分，不产生梯度回传，彻底消除辅助损失对模型能力的损害。
5. ReAct 状态机闭环: Thought 思考 -> Action 工具调用 -> Observation 环境反馈 -> Final Answer。
```